# fMRI Longitudinal Pipeline: Long to Wide + Merge

This notebook creates a reproducible pipeline to:
- Convert motion metrics long -> wide (`T1`, `T2`, and variable suffixes `_T1`/`_T2`).
- Convert contrasts long -> wide using ROI-based columns (`ROI_metric_T1`/`ROI_metric_T2`).
- Match (merge) motion and contrasts into one final participant-level table.
- Export all outputs to one Excel workbook.

In [24]:
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 300)

# Inputs
MOTION_FILE = Path(r"C:\Users\okkam\Desktop\labo\article 2\Longitudinal_Multimodal_Data_CIMAQ\fMRI\ALL_MOTION_METRICS.xlsx")
CONTRASTS_FILE = Path(r"C:\Users\okkam\Desktop\labo\article 2\Longitudinal_Multimodal_Data_CIMAQ\fMRI\119_contrasts.xlsx")

# Output
OUT_FILE = MOTION_FILE.parent / "long_to_wide_fMRI_outputs.xlsx"

MOTION_FILE, CONTRASTS_FILE, OUT_FILE

(WindowsPath('C:/Users/okkam/Desktop/labo/article 2/Longitudinal_Multimodal_Data_CIMAQ/fMRI/ALL_MOTION_METRICS.xlsx'),
 WindowsPath('C:/Users/okkam/Desktop/labo/article 2/Longitudinal_Multimodal_Data_CIMAQ/fMRI/119_contrasts.xlsx'),
 WindowsPath('C:/Users/okkam/Desktop/labo/article 2/Longitudinal_Multimodal_Data_CIMAQ/fMRI/long_to_wide_fMRI_outputs.xlsx'))

In [25]:
def _auto_find_id_session_columns(df: pd.DataFrame):
    cols = list(df.columns)
    lower_map = {c.lower().strip(): c for c in cols}

    id_candidates = [
        lower_map.get("id"),
        lower_map.get("participant_id"),
        lower_map.get("subject"),
        cols[0] if len(cols) > 0 else None,
    ]
    ses_candidates = [
        lower_map.get("session"),
        lower_map.get("visit"),
        cols[1] if len(cols) > 1 else None,
    ]

    id_col = next((c for c in id_candidates if c in df.columns), None)
    session_col = next((c for c in ses_candidates if c in df.columns), None)

    if id_col is None or session_col is None:
        raise ValueError("Could not identify ID and Session columns.")

    return id_col, session_col


def _session_numeric_order(series: pd.Series) -> pd.Series:
    extracted = series.astype(str).str.extract(r"(\d+)", expand=False)
    return pd.to_numeric(extracted, errors="coerce").fillna(9999)


def _clean_label(value: str) -> str:
    x = str(value).strip()
    x = re.sub(r"\s+", "_", x)
    x = re.sub(r"[^A-Za-z0-9_.-]", "", x)
    return x


def long_to_wide_two_timepoints(df: pd.DataFrame, id_col: str, session_col: str, variable_columns=None) -> pd.DataFrame:
    x = df.copy()
    if variable_columns is None:
        variable_columns = [c for c in x.columns if c not in [id_col, session_col]]

    x = x.dropna(subset=[id_col, session_col]).copy()
    x["_session_num"] = _session_numeric_order(x[session_col])
    x = x.sort_values([id_col, "_session_num", session_col])
    x["_tp"] = x.groupby(id_col).cumcount() + 1
    x = x[x["_tp"].isin([1, 2])].copy()

    session_wide = (
        x.pivot_table(index=id_col, columns="_tp", values=session_col, aggfunc="first")
        .rename(columns={1: "T1", 2: "T2"})
    )

    parts = [session_wide]
    for var in variable_columns:
        p = x.pivot_table(index=id_col, columns="_tp", values=var, aggfunc="first")
        p = p.rename(columns={1: f"{var}_T1", 2: f"{var}_T2"})
        parts.append(p)

    out = pd.concat(parts, axis=1).reset_index()

    t1_vars = [f"{v}_T1" for v in variable_columns if f"{v}_T1" in out.columns]
    t2_vars = [f"{v}_T2" for v in variable_columns if f"{v}_T2" in out.columns]
    final_cols = [id_col, "T1"] + t1_vars + ["T2"] + t2_vars
    final_cols = [c for c in final_cols if c in out.columns]
    return out[final_cols]


def classify_roi_name(roi_name: str) -> str:
    v = str(roi_name).strip().lower()
    if "hippocampus" in v:
        return "01_hippocampus"
    if "parietal_sup" in v or "superior_pariet" in v:
        return "02_superior_parietal_cortex"
    if any(k in v for k in ["parahipp", "entorh", "subiculum", "fornix", "fimbria", "hippocampal"]):
        return "03_hippocampus_related"
    if "precuneus" in v:
        return "04_precuneus"
    if any(k in v for k in ["cingulum_ant", "acc", "anterior_cing"]):
        return "05_acc"
    return "99_rest"


def long_to_wide_contrasts_by_roi(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()
    x.columns = [c.strip() for c in x.columns]

    id_col, session_col = _auto_find_id_session_columns(x)
    roi_col = [c for c in x.columns if "roi" in c.lower()][0]
    x[roi_col] = x[roi_col].astype(str).str.strip()

    metric_cols = [c for c in x.columns if c not in [id_col, session_col, roi_col]]

    ses_map = x[[id_col, session_col]].drop_duplicates().copy()
    ses_map["_session_num"] = _session_numeric_order(ses_map[session_col])
    ses_map = ses_map.sort_values([id_col, "_session_num", session_col])
    ses_map["_tp"] = ses_map.groupby(id_col).cumcount() + 1
    ses_map = ses_map[ses_map["_tp"].isin([1, 2])]

    x = x.merge(ses_map[[id_col, session_col, "_tp"]], on=[id_col, session_col], how="inner")

    session_wide = (
        ses_map.pivot_table(index=id_col, columns="_tp", values=session_col, aggfunc="first")
        .rename(columns={1: "T1", 2: "T2"})
    )

    parts = [session_wide]
    for metric in metric_cols:
        p = x.pivot_table(index=id_col, columns=["_tp", roi_col], values=metric, aggfunc="first")
        p.columns = [
            f"{_clean_label(roi)}_{_clean_label(metric)}_T{int(tp)}"
            for tp, roi in p.columns
        ]
        parts.append(p)

    out = pd.concat(parts, axis=1).reset_index()

    t1_cols = [c for c in out.columns if str(c).endswith("_T1")]
    t2_cols = [c for c in out.columns if str(c).endswith("_T2")]

    def _sort_key(col_name: str):
        base = col_name.rsplit("_T", 1)[0]
        parts = base.split("_")
        metric = parts[-1] if parts else ""
        roi = "_".join(parts[:-1]) if len(parts) > 1 else base
        return (classify_roi_name(roi), roi.lower(), metric.lower())

    t1_cols = sorted(t1_cols, key=_sort_key)
    t2_cols = sorted(t2_cols, key=_sort_key)

    final_cols = [id_col, "T1"] + t1_cols + ["T2"] + t2_cols
    final_cols = [c for c in final_cols if c in out.columns]
    return out[final_cols]


print("Helpers loaded.")

Helpers loaded.


In [26]:
# 1) Motion: long -> wide
motion_long = pd.read_excel(MOTION_FILE)
id_col_m, ses_col_m = _auto_find_id_session_columns(motion_long)
motion_vars = [c for c in motion_long.columns if c not in [id_col_m, ses_col_m]]

motion_wide = long_to_wide_two_timepoints(
    motion_long,
    id_col=id_col_m,
    session_col=ses_col_m,
    variable_columns=motion_vars,
)

print("Motion long:", motion_long.shape)
print("Motion wide:", motion_wide.shape)
motion_wide.head()

Motion long: (139, 15)
Motion wide: (74, 29)


_tp,ID,T1,TD_mean_T1,TD_STD_T1,TD_median_T1,TD_max_T1,TD_>1mm_T1,TD_>2mm_T1,TD_>3mm_T1,STS_mean_T1,STS_STD_T1,STS_median_T1,STS_max_T1,STS_>0.5mm_T1,STS_>1mm_T1,T2,TD_mean_T2,TD_STD_T2,TD_median_T2,TD_max_T2,TD_>1mm_T2,TD_>2mm_T2,TD_>3mm_T2,STS_mean_T2,STS_STD_T2,STS_median_T2,STS_max_T2,STS_>0.5mm_T2,STS_>1mm_T2
0,3002498,t04,0.4579,0.1482,0.4989,0.7831,0.0,0.0,0.0,0.1565,0.1132,0.1324,1.1124,4.0,1.0,t06,0.2214,0.0826,0.2172,0.4921,0.0,0.0,0.0,0.1950,0.1154,0.1727,1.2546,6.0,1.0
1,3025432,t02,0.8227,0.3557,0.9174,1.4089,284.0,0.0,0.0,0.2220,0.2880,0.1585,2.8437,29.0,15.0,t06,0.8135,0.2308,0.8608,1.5575,112.0,0.0,0.0,0.1848,0.2564,0.1360,2.9957,17.0,11.0
2,3100205,t04,0.9387,0.2679,1.0113,1.3353,320.0,0.0,0.0,0.2841,0.1428,0.2588,2.6461,23.0,1.0,t08,0.6674,0.1899,0.6966,1.1379,16.0,0.0,0.0,0.3701,0.1898,0.3464,2.6835,127.0,6.0
3,3123186,t02,0.5541,0.2232,0.6061,1.0273,6.0,0.0,0.0,0.1965,0.1636,0.1386,1.6715,37.0,1.0,t04,0.1691,0.0976,0.1458,0.5049,0.0,0.0,0.0,0.2641,0.2057,0.1803,1.0529,54.0,1.0
4,3149469,t04,0.3443,0.1434,0.3848,0.6536,0.0,0.0,0.0,0.1342,0.0493,0.1312,0.3103,0.0,0.0,t06,0.4423,0.2374,0.4952,0.9153,0.0,0.0,0.0,0.1949,0.0819,0.1873,0.7871,4.0,0.0


In [27]:
# 2) Contrasts: long -> wide (ROI-based)
contrasts_long = pd.read_excel(CONTRASTS_FILE)
contrasts_wide = long_to_wide_contrasts_by_roi(contrasts_long)

x_tmp = contrasts_long.copy()
x_tmp.columns = [c.strip() for c in x_tmp.columns]
roi_col_tmp = [c for c in x_tmp.columns if "roi" in c.lower()][0]
group_counts = (
    x_tmp[roi_col_tmp]
    .astype(str)
    .str.strip()
    .map(classify_roi_name)
    .value_counts()
    .sort_index()
)

print("Contrasts long:", contrasts_long.shape)
print("Contrasts wide:", contrasts_wide.shape)
print("\nROI counts by custom group:")
print(group_counts)
contrasts_wide.head()

Contrasts long: (9684, 7)
Contrasts wide: (71, 659)

ROI counts by custom group:
ROI name
01_hippocampus                  238
02_superior_parietal_cortex     230
03_hippocampus_related          236
04_precuneus                    238
05_acc                          237
99_rest                        8505
Name: count, dtype: int64


,ID,T1,Hippocampus_L_Average_T1,Hippocampus_L_Size_T1,Hippocampus_L_Std.Dev._T1,Hippocampus_L_T_T1,Hippocampus_R_Average_T1,Hippocampus_R_Size_T1,Hippocampus_R_Std.Dev._T1,Hippocampus_R_T_T1,Parietal_Sup_L_Average_T1,Parietal_Sup_L_Size_T1,Parietal_Sup_L_Std.Dev._T1,Parietal_Sup_L_T_T1,Parietal_Sup_R_Average_T1,Parietal_Sup_R_Size_T1,Parietal_Sup_R_Std.Dev._T1,Parietal_Sup_R_T_T1,ParaHippocampal_L_Average_T1,ParaHippocampal_L_Size_T1,ParaHippocampal_L_Std.Dev._T1,ParaHippocampal_L_T_T1,ParaHippocampal_R_Average_T1,ParaHippocampal_R_Size_T1,ParaHippocampal_R_Std.Dev._T1,ParaHippocampal_R_T_T1,Precuneus_L_Average_T1,Precuneus_L_Size_T1,Precuneus_L_Std.Dev._T1,Precuneus_L_T_T1,Precuneus_R_Average_T1,Precuneus_R_Size_T1,Precuneus_R_Std.Dev._T1,Precuneus_R_T_T1,Cingulum_Ant_L_Average_T1,Cingulum_Ant_L_Size_T1,Cingulum_Ant_L_Std.Dev._T1,Cingulum_Ant_L_T_T1,Cingulum_Ant_R_Average_T1,Cingulum_Ant_R_Size_T1,Cingulum_Ant_R_Std.Dev._T1,Cingulum_Ant_R_T_T1,Amygdala_L_Average_T1,Amygdala_L_Size_T1,Amygdala_L_Std.Dev._T1,Amygdala_L_T_T1,Amygdala_R_Average_T1,Amygdala_R_Size_T1,Amygdala_R_Std.Dev._T1,Amygdala_R_T_T1,Angular_L_Average_T1,Angular_L_Size_T1,Angular_L_Std.Dev._T1,Angular_L_T_T1,Angular_R_Average_T1,Angular_R_Size_T1,Angular_R_Std.Dev._T1,Angular_R_T_T1,Calcarine_L_Average_T1,Calcarine_L_Size_T1,Calcarine_L_Std.Dev._T1,Calcarine_L_T_T1,Calcarine_R_Average_T1,Calcarine_R_Size_T1,Calcarine_R_Std.Dev._T1,Calcarine_R_T_T1,Cingulum_Mid_L_Average_T1,Cingulum_Mid_L_Size_T1,Cingulum_Mid_L_Std.Dev._T1,Cingulum_Mid_L_T_T1,Cingulum_Mid_R_Average_T1,Cingulum_Mid_R_Size_T1,Cingulum_Mid_R_Std.Dev._T1,Cingulum_Mid_R_T_T1,Cingulum_Post_L_Average_T1,Cingulum_Post_L_Size_T1,Cingulum_Post_L_Std.Dev._T1,Cingulum_Post_L_T_T1,Cingulum_Post_R_Average_T1,Cingulum_Post_R_Size_T1,Cingulum_Post_R_Std.Dev._T1,Cingulum_Post_R_T_T1,Cuneus_L_Average_T1,Cuneus_L_Size_T1,Cuneus_L_Std.Dev._T1,Cuneus_L_T_T1,Cuneus_R_Average_T1,Cuneus_R_Size_T1,Cuneus_R_Std.Dev._T1,Cuneus_R_T_T1,Frontal_Inf_Oper_L_Average_T1,Frontal_Inf_Oper_L_Size_T1,Frontal_Inf_Oper_L_Std.Dev._T1,Frontal_Inf_Oper_L_T_T1,Frontal_Inf_Oper_R_Average_T1,Frontal_Inf_Oper_R_Size_T1,Frontal_Inf_Oper_R_Std.Dev._T1,Frontal_Inf_Oper_R_T_T1,Frontal_Inf_Orb_L_Average_T1,Frontal_Inf_Orb_L_Size_T1,Frontal_Inf_Orb_L_Std.Dev._T1,Frontal_Inf_Orb_L_T_T1,Frontal_Inf_Orb_R_Average_T1,Frontal_Inf_Orb_R_Size_T1,Frontal_Inf_Orb_R_Std.Dev._T1,Frontal_Inf_Orb_R_T_T1,Frontal_Inf_Tri_L_Average_T1,Frontal_Inf_Tri_L_Size_T1,Frontal_Inf_Tri_L_Std.Dev._T1,Frontal_Inf_Tri_L_T_T1,Frontal_Inf_Tri_R_Average_T1,Frontal_Inf_Tri_R_Size_T1,Frontal_Inf_Tri_R_Std.Dev._T1,Frontal_Inf_Tri_R_T_T1,Frontal_Med_Orb_L_Average_T1,Frontal_Med_Orb_L_Size_T1,Frontal_Med_Orb_L_Std.Dev._T1,Frontal_Med_Orb_L_T_T1,Frontal_Med_Orb_R_Average_T1,Frontal_Med_Orb_R_Size_T1,Frontal_Med_Orb_R_Std.Dev._T1,Frontal_Med_Orb_R_T_T1,Frontal_Mid_L_Average_T1,Frontal_Mid_L_Size_T1,Frontal_Mid_L_Std.Dev._T1,Frontal_Mid_L_T_T1,Frontal_Mid_Orb_L_Average_T1,Frontal_Mid_Orb_L_Size_T1,Frontal_Mid_Orb_L_Std.Dev._T1,Frontal_Mid_Orb_L_T_T1,Frontal_Mid_Orb_R_Average_T1,Frontal_Mid_Orb_R_Size_T1,Frontal_Mid_Orb_R_Std.Dev._T1,Frontal_Mid_Orb_R_T_T1,Frontal_Mid_R_Average_T1,Frontal_Mid_R_Size_T1,Frontal_Mid_R_Std.Dev._T1,Frontal_Mid_R_T_T1,Frontal_Sup_L_Average_T1,Frontal_Sup_L_Size_T1,Frontal_Sup_L_Std.Dev._T1,Frontal_Sup_L_T_T1,Frontal_Sup_Medial_L_Average_T1,Frontal_Sup_Medial_L_Size_T1,Frontal_Sup_Medial_L_Std.Dev._T1,Frontal_Sup_Medial_L_T_T1,Frontal_Sup_Medial_R_Average_T1,Frontal_Sup_Medial_R_Size_T1,Frontal_Sup_Medial_R_Std.Dev._T1,Frontal_Sup_Medial_R_T_T1,...,Insula_L_Std.Dev._T2,Insula_L_T_T2,Insula_R_Average_T2,Insula_R_Size_T2,Insula_R_Std.Dev._T2,Insula_R_T_T2,Lingual_L_Average_T2,Lingual_L_Size_T2,Lingual_L_Std.Dev._T2,Lingual_L_T_T2,Lingual_R_Average_T2,Lingual_R_Size_T2,Lingual_R_Std.Dev._T2,Lingual_R_T_T2,Occipital_Inf_L_Average_T2,Occipital_Inf_L_Size_T2,Occipital_Inf_L_Std.Dev._T2,Occipital_Inf_L_T_T2,Occipital_Inf_R_Average_T2,Occipital_Inf_R_Size_T2,Occipit

In [28]:
# 3) Match motion + contrasts together (merge)
# Merge by ID and reconcile session columns from both sources.
id_col = "ID" if "ID" in motion_wide.columns else motion_wide.columns[0]

merged = motion_wide.merge(
    contrasts_wide,
    on=id_col,
    how="outer",
    suffixes=("_motion", "_contrasts"),
)

# Reconcile T1/T2 session labels if duplicated by merge
if "T1_motion" in merged.columns and "T1_contrasts" in merged.columns:
    merged["T1"] = merged["T1_motion"].combine_first(merged["T1_contrasts"])
    merged["T1_mismatch"] = (
        merged["T1_motion"].notna()
        & merged["T1_contrasts"].notna()
        & (merged["T1_motion"] != merged["T1_contrasts"])
    )

if "T2_motion" in merged.columns and "T2_contrasts" in merged.columns:
    merged["T2"] = merged["T2_motion"].combine_first(merged["T2_contrasts"])
    merged["T2_mismatch"] = (
        merged["T2_motion"].notna()
        & merged["T2_contrasts"].notna()
        & (merged["T2_motion"] != merged["T2_contrasts"])
    )

# Keep clean session columns first
drop_cols = [c for c in ["T1_motion", "T1_contrasts", "T2_motion", "T2_contrasts"] if c in merged.columns]
merged = merged.drop(columns=drop_cols)

front = [c for c in [id_col, "T1", "T2", "T1_mismatch", "T2_mismatch"] if c in merged.columns]
rest = [c for c in merged.columns if c not in front]
merged_wide = merged[front + rest]

print("Merged wide:", merged_wide.shape)
print("T1 mismatch count:", int(merged_wide.get("T1_mismatch", pd.Series(dtype=bool)).sum()))
print("T2 mismatch count:", int(merged_wide.get("T2_mismatch", pd.Series(dtype=bool)).sum()))
merged_wide.head()

Merged wide: (74, 687)
T1 mismatch count: 2
T2 mismatch count: 0


,ID,T1,T2,T1_mismatch,T2_mismatch,TD_mean_T1,TD_STD_T1,TD_median_T1,TD_max_T1,TD_>1mm_T1,TD_>2mm_T1,TD_>3mm_T1,STS_mean_T1,STS_STD_T1,STS_median_T1,STS_max_T1,STS_>0.5mm_T1,STS_>1mm_T1,TD_mean_T2,TD_STD_T2,TD_median_T2,TD_max_T2,TD_>1mm_T2,TD_>2mm_T2,TD_>3mm_T2,STS_mean_T2,STS_STD_T2,STS_median_T2,STS_max_T2,STS_>0.5mm_T2,STS_>1mm_T2,Hippocampus_L_Average_T1,Hippocampus_L_Size_T1,Hippocampus_L_Std.Dev._T1,Hippocampus_L_T_T1,Hippocampus_R_Average_T1,Hippocampus_R_Size_T1,Hippocampus_R_Std.Dev._T1,Hippocampus_R_T_T1,Parietal_Sup_L_Average_T1,Parietal_Sup_L_Size_T1,Parietal_Sup_L_Std.Dev._T1,Parietal_Sup_L_T_T1,Parietal_Sup_R_Average_T1,Parietal_Sup_R_Size_T1,Parietal_Sup_R_Std.Dev._T1,Parietal_Sup_R_T_T1,ParaHippocampal_L_Average_T1,ParaHippocampal_L_Size_T1,ParaHippocampal_L_Std.Dev._T1,ParaHippocampal_L_T_T1,ParaHippocampal_R_Average_T1,ParaHippocampal_R_Size_T1,ParaHippocampal_R_Std.Dev._T1,ParaHippocampal_R_T_T1,Precuneus_L_Average_T1,Precuneus_L_Size_T1,Precuneus_L_Std.Dev._T1,Precuneus_L_T_T1,Precuneus_R_Average_T1,Precuneus_R_Size_T1,Precuneus_R_Std.Dev._T1,Precuneus_R_T_T1,Cingulum_Ant_L_Average_T1,Cingulum_Ant_L_Size_T1,Cingulum_Ant_L_Std.Dev._T1,Cingulum_Ant_L_T_T1,Cingulum_Ant_R_Average_T1,Cingulum_Ant_R_Size_T1,Cingulum_Ant_R_Std.Dev._T1,Cingulum_Ant_R_T_T1,Amygdala_L_Average_T1,Amygdala_L_Size_T1,Amygdala_L_Std.Dev._T1,Amygdala_L_T_T1,Amygdala_R_Average_T1,Amygdala_R_Size_T1,Amygdala_R_Std.Dev._T1,Amygdala_R_T_T1,Angular_L_Average_T1,Angular_L_Size_T1,Angular_L_Std.Dev._T1,Angular_L_T_T1,Angular_R_Average_T1,Angular_R_Size_T1,Angular_R_Std.Dev._T1,Angular_R_T_T1,Calcarine_L_Average_T1,Calcarine_L_Size_T1,Calcarine_L_Std.Dev._T1,Calcarine_L_T_T1,Calcarine_R_Average_T1,Calcarine_R_Size_T1,Calcarine_R_Std.Dev._T1,Calcarine_R_T_T1,Cingulum_Mid_L_Average_T1,Cingulum_Mid_L_Size_T1,Cingulum_Mid_L_Std.Dev._T1,Cingulum_Mid_L_T_T1,Cingulum_Mid_R_Average_T1,Cingulum_Mid_R_Size_T1,Cingulum_Mid_R_Std.Dev._T1,Cingulum_Mid_R_T_T1,Cingulum_Post_L_Average_T1,Cingulum_Post_L_Size_T1,Cingulum_Post_L_Std.Dev._T1,Cingulum_Post_L_T_T1,Cingulum_Post_R_Average_T1,Cingulum_Post_R_Size_T1,Cingulum_Post_R_Std.Dev._T1,Cingulum_Post_R_T_T1,Cuneus_L_Average_T1,Cuneus_L_Size_T1,Cuneus_L_Std.Dev._T1,Cuneus_L_T_T1,Cuneus_R_Average_T1,Cuneus_R_Size_T1,Cuneus_R_Std.Dev._T1,Cuneus_R_T_T1,Frontal_Inf_Oper_L_Average_T1,Frontal_Inf_Oper_L_Size_T1,Frontal_Inf_Oper_L_Std.Dev._T1,Frontal_Inf_Oper_L_T_T1,Frontal_Inf_Oper_R_Average_T1,Frontal_Inf_Oper_R_Size_T1,Frontal_Inf_Oper_R_Std.Dev._T1,Frontal_Inf_Oper_R_T_T1,Frontal_Inf_Orb_L_Average_T1,Frontal_Inf_Orb_L_Size_T1,Frontal_Inf_Orb_L_Std.Dev._T1,Frontal_Inf_Orb_L_T_T1,Frontal_Inf_Orb_R_Average_T1,Frontal_Inf_Orb_R_Size_T1,Frontal_Inf_Orb_R_Std.Dev._T1,Frontal_Inf_Orb_R_T_T1,Frontal_Inf_Tri_L_Average_T1,Frontal_Inf_Tri_L_Size_T1,Frontal_Inf_Tri_L_Std.Dev._T1,Frontal_Inf_Tri_L_T_T1,Frontal_Inf_Tri_R_Average_T1,Frontal_Inf_Tri_R_Size_T1,Frontal_Inf_Tri_R_Std.Dev._T1,Frontal_Inf_Tri_R_T_T1,Frontal_Med_Orb_L_Average_T1,Frontal_Med_Orb_L_Size_T1,Frontal_Med_Orb_L_Std.Dev._T1,Frontal_Med_Orb_L_T_T1,Frontal_Med_Orb_R_Average_T1,Frontal_Med_Orb_R_Size_T1,Frontal_Med_Orb_R_Std.Dev._T1,...,Insula_L_Std.Dev._T2,Insula_L_T_T2,Insula_R_Average_T2,Insula_R_Size_T2,Insula_R_Std.Dev._T2,Insula_R_T_T2,Lingual_L_Average_T2,Lingual_L_Size_T2,Lingual_L_Std.Dev._T2,Lingual_L_T_T2,Lingual_R_Average_T2,Lingual_R_Size_T2,Lingual_R_Std.Dev._T2,Lingual_R_T_T2,Occipital_Inf_L_Average_T2,Occipital_Inf_L_Size_T2,Occipital_Inf_L_Std.Dev._T2,Occipital_Inf_L_T_T2,Occipital_Inf_R_Average_T2,Occipital_Inf_R_Size_T2,Occipital_Inf_R_Std.Dev._T2,Occipital_Inf_R_T_T2,Occipital_Mid_L_Average_T2,Occipital_Mid_L_Size_T2,Occipital_Mid_L_Std.Dev._T2,Occipital_Mid_L_T_T2,Occipital_Mid_R_Average_T2,Occipital_Mid_R_Size_T2,Occipital_Mid_R_Std.Dev._T2,Occipital_Mid_R_T_T2,Occipital_Sup_L_Average_T2,Occipital_Sup_L_Size_T2,Occipital_Sup_L_Std.Dev._T2,Occipital_Sup_L_T_T2,Occipital_Sup_R_Average_T2,Occipital_Sup_R_Size_T2,Occipital_Sup_R_Std.Dev._T2,Occip

In [29]:
# 4) Export all outputs
with pd.ExcelWriter(OUT_FILE, engine="openpyxl") as writer:
    motion_wide.to_excel(writer, sheet_name="ALL_MOTION_WIDE", index=False)
    contrasts_wide.to_excel(writer, sheet_name="CONTRASTS_WIDE", index=False)
    merged_wide.to_excel(writer, sheet_name="MERGED_WIDE", index=False)

print(f"Saved workbook: {OUT_FILE}")

Saved workbook: C:\Users\okkam\Desktop\labo\article 2\Longitudinal_Multimodal_Data_CIMAQ\fMRI\long_to_wide_fMRI_outputs.xlsx
